<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/GAN-WK-11.A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install required libraries
!pip install -q transformers[torch] datasets

import torch
import os
import shutil
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

# 2. Prepare Product Description Dataset (Input: Features -> Output: Description)
product_data = [
    {"text": "Product: Wireless Headphones, Features: Noise Cancelling, Bluetooth 5.0, 20h Battery. Description: Experience pure sound with our latest noise-cancelling headphones. With seamless Bluetooth 5.0 connectivity and a massive 20-hour battery life, your music never has to stop."},
    {"text": "Product: Running Shoes, Features: Lightweight, Breathable Mesh, Shock Absorption. Description: Reach your personal best with these ultra-lightweight running shoes. The breathable mesh keeps your feet cool, while advanced shock absorption protects your joints on every stride."},
    {"text": "Product: Coffee Maker, Features: Programmable Timer, 12-Cup Capacity, Stainless Steel. Description: Wake up to the perfect brew every morning. This sleek stainless steel coffee maker features a 12-cup capacity and a programmable timer for your convenience."},
    {"text": "Product: Smartwatch, Features: Heart Rate Monitor, GPS, Waterproof. Description: Stay on top of your fitness goals with this rugged smartwatch. Featuring a precise heart rate monitor and built-in GPS, it is completely waterproof and ready for any adventure."}
]

raw_dataset = Dataset.from_dict({"text": [item["text"] for item in product_data]})

# 3. Load Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 4. Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Setup and Train
output_dir = "./gpt2-product-gen"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

model = GPT2LMHeadModel.from_pretrained(model_name)
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=30, # Higher epochs for a tiny dataset to ensure it learns the pattern
    per_device_train_batch_size=1,
    save_strategy="no",
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

print("Conditioning model for product descriptions...")
trainer.train()

# 6. Generate for the specific requirement
prompt = "Product: Smartwatch, Features: Heart Rate Monitor, GPS, Waterproof. Description:"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids,
    max_length=100,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    repetition_penalty=1.5, # Prevents the model from just repeating the features
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print("\n--- GENERATED PRODUCT DESCRIPTION ---")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Conditioning model for product descriptions...


Step,Training Loss
5,3.162915
10,2.052895
15,1.396981
20,1.023514
25,0.734597
30,0.422771
35,0.368098
40,0.237173
45,0.213006
50,0.102124



--- GENERATED PRODUCT DESCRIPTION ---
Product: Smartwatch, Features: Heart Rate Monitor, GPS, Waterproof. Description: Stay on top of your fitness goals with this rugged smart watch! Featuring a precise heart rate monitor and built-in water proof strap for every day operation – it is completely waterproof & ready to go when you need something new or fun while producing impressive running speed.." – Apple "AIO."


Start Your Own Road Trip With This Wireless Headlamp Pro Series Running Shoes that are breathable enough to
